# Instacart Reorder Prediction (Deep Learning)
This notebook uses a subset of the Instacart dataset and builds a simple embedding-based neural network.

In [22]:

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Concatenate
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split


## Create Sample Dataset (Subset Simulation)

In [23]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yasserh/instacart-online-grocery-basket-analysis-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'instacart-online-grocery-basket-analysis-dataset' dataset.
Path to dataset files: /kaggle/input/instacart-online-grocery-basket-analysis-dataset


In [3]:
import pandas as pd

orders = pd.read_csv(path+"/"+"orders.csv")
prior = pd.read_csv(path+"/"+"order_products__prior.csv")

# Sample 10,000 users
sample_users = orders['user_id'].drop_duplicates().sample(10000, random_state=42)

orders_sample = orders[orders['user_id'].isin(sample_users)]
prior_sample = prior[prior['order_id'].isin(orders_sample['order_id'])]

In [4]:
orders.shape

(3421083, 7)

## Preprocessing

In [5]:
products = pd.read_csv(path+"/"+"products.csv")
aisles = pd.read_csv(path+"/"+"aisles.csv")
departments = pd.read_csv(path+"/"+"departments.csv")

# Merge product info
prod_merged = products.merge(aisles, on='aisle_id')\
                      .merge(departments, on='department_id')

# Merge with order data
df = prior_sample.merge(orders_sample, on='order_id')\
                 .merge(prod_merged, on='product_id')

In [6]:
df.isnull().sum()

,0
order_id,0
product_id,0
add_to_cart_order,0
reordered,0
user_id,0
eval_set,0
order_number,0
order_dow,0
order_hour_of_day,0
days_since_prior_order,100799


In [7]:
user_features = df.groupby('user_id').agg({
    'order_id': 'nunique',
    'add_to_cart_order': 'mean'
}).rename(columns={
    'order_id': 'total_orders',
    'add_to_cart_order': 'avg_cart_position'
})

In [8]:
product_features = df.groupby('product_id').agg({
    'order_id': 'count',
    'reordered': 'mean'
}).rename(columns={
    'order_id': 'product_popularity',
    'reordered': 'reorder_rate'
})

In [9]:
user_features

,total_orders,avg_cart_position
user_id,,
5,4,5.513514
7,20,7.252427
13,12,4.148148
34,5,5.218750
71,23,11.473684
...,...,...
206126,29,11.477099
206133,4,4.619048
206138,14,4.965812


In [10]:
user_product = df.groupby(['user_id', 'product_id']).size()\
                 .reset_index(name='user_product_count')

In [11]:
df = df.merge(user_features, on='user_id')\
       .merge(product_features, on='product_id')\
       .merge(user_product, on=['user_id', 'product_id'])

In [12]:
df['is_weekend'] = df['order_dow'].apply(lambda x: 1 if x in [0,6] else 0)

def time_bucket(hour):
    if hour < 12:
        return 'morning'
    elif hour < 18:
        return 'afternoon'
    else:
        return 'evening'

df['time_of_day'] = df['order_hour_of_day'].apply(time_bucket)

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1578962 entries, 0 to 1578961
Data columns (total 22 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   order_id                1578962 non-null  int64  
 1   product_id              1578962 non-null  int64  
 2   add_to_cart_order       1578962 non-null  int64  
 3   reordered               1578962 non-null  int64  
 4   user_id                 1578962 non-null  int64  
 5   eval_set                1578962 non-null  object 
 6   order_number            1578962 non-null  int64  
 7   order_dow               1578962 non-null  int64  
 8   order_hour_of_day       1578962 non-null  int64  
 9   days_since_prior_order  1478163 non-null  float64
 10  product_name            1578962 non-null  object 
 11  aisle_id                1578962 non-null  int64  
 12  department_id           1578962 non-null  int64  
 13  aisle                   1578962 non-null  object 
 14  de

In [14]:
df.drop(['eval_set','days_since_prior_order','product_name'],axis=1,inplace=True)

In [15]:
df = pd.get_dummies(df, columns=['aisle', 'department', 'time_of_day'], drop_first=True)

In [16]:
df


,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,aisle_id,department_id,...,department_meat seafood,department_missing,department_other,department_pantry,department_personal care,department_pets,department_produce,department_snacks,time_of_day_evening,time_of_day_morning
0,28,35108,1,0,98256,29,3,13,36,16,...,False,False,False,False,False,False,False,False,False,False
1,28,40593,2,1,98256,29,3,13,108,16,...,False,False,False,False,False,False,False,False,False,False
2,28,17461,3,0,98256,29,3,13,35,12,...,True,False,False,False,False,False,False,False,False,False
3,28,22825,4,1,98256,29,3,13,24,4,...,False,False,False,False,False,False,True,False,False,False
4,28,25256,5,1,98256,29,3,13,84,16,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1578957,3421083,39678,6,1,25247,24,2,6,74,17,...,False,False,False,False,False,False,False,False,False,True
1578958,3421083,11352,7,0,25247,24,2,6,78,19,...,False,False,False,False,False,False,False,True,False,True
1578959,3421083,4600,8,0,25247,24,2,6,52,1,...,False,False,False,False,False,False,False,False,False,True
1578960,3421083,24852,9,1,25247,24,2,6,24,4,...,False,False,False,False,False,False,True,False,False,True


In [17]:
from sklearn.model_selection import train_test_split

X = df.drop(['reordered'], axis=1)
y = df['reordered']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [19]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_pred = model.predict(X_test)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

In [20]:
print(precision,recall,f1)

0.862546472888803 0.9606319536682029 0.9089507546944207


In [26]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Concatenate
from tensorflow.keras.models import Model

# Example sizes
n_users = df['user_id'].nunique()
n_products = df['product_id'].nunique()

# Inputs
user_input = Input(shape=(1,))
product_input = Input(shape=(1,))
num_input = Input(shape=(X_train_num.shape[1],))  # numerical features

# Embeddings
user_emb = Embedding(input_dim=n_users, output_dim=16)(user_input)
product_emb = Embedding(input_dim=n_products, output_dim=16)(product_input)

# Flatten
user_vec = Flatten()(user_emb)
product_vec = Flatten()(product_emb)

# Combine
x = Concatenate()([user_vec, product_vec, num_input])

# Dense layers
x = Dense(64, activation='relu')(x)
x = Dense(32, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

# Model
model = Model(inputs=[user_input, product_input, num_input], outputs=output)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [24]:
numerical_features = X_train.drop(columns=['user_id', 'product_id', 'order_id']).columns
X_train_num = X_train[numerical_features].astype(np.float32)
X_test_num = X_test[numerical_features].astype(np.float32)

# Prepare inputs for the deep learning model
X_train_inputs = [
    X_train['user_id'].values,
    X_train['product_id'].values,
    X_train_num.values
]

X_test_inputs = [
    X_test['user_id'].values,
    X_test['product_id'].values,
    X_test_num.values
]

y_train_dl = y_train.values
y_test_dl = y_test.values

In [27]:
model.fit(
    X_train_inputs,
    y_train_dl,
    epochs=5,
    batch_size=256,
    validation_data=(X_test_inputs, y_test_dl)
)

Epoch 1/5
4935/4935 ━━━━━━━━━━━━━━━━━━━━ 22s 4ms/step - accuracy: 0.8277 - loss: 0.6033 - val_accuracy: 0.8616 - val_loss: 0.3962
Epoch 2/5
4935/4935 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - accuracy: 0.8686 - loss: 0.3545 - val_accuracy: 0.8630 - val_loss: 0.4893
Epoch 3/5
4935/4935 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - accuracy: 0.8781 - loss: 0.3057 - val_accuracy: 0.8884 - val_loss: 0.2394
Epoch 4/5
4935/4935 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - accuracy: 0.8843 - loss: 0.2651 - val_accuracy: 0.8894 - val_loss: 0.2450
Epoch 5/5
4935/4935 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - accuracy: 0.8873 - loss: 0.2492 - val_accuracy: 0.8804 - val_loss: 0.2687
